In [7]:
import numpy as np
from tqdm import tqdm
from typing import Callable
from scipy.sparse import diags_array
from scipy.integrate import solve_ivp

## Korteweg-De Vries equation

The Korteweg-De Vries (KdV) equation is a PDE which serves as a mathematical model of waves on shallow water surfaces. It models one dimensional nonlinear dispersive nondissipative waves described by a function $u(t,x)$ that satisfies the equation

$$u_t+\eta uu_x-\gamma^2u_{xxx}=0 $$

We let $\eta=6$ and $\gamma=1$ and assume periodic solutions $u(0,t)=u(P,t)$ on the domain $[0,P]$, with $P=20$. 

In [2]:
# Recommended wave-specific parameters.
eta = 6.0
gamma = 1.0

# Parameters for the spatial dimension
num_spatial_points = 100
period = 20

# Temporal parameters
timestep = 0.0025
max_time = 1
num_trajectories = 1

x = np.linspace(0, period - period / num_spatial_points, num_spatial_points)
t_axis = np.arange(0, max_time, timestep)
M = x.size
dx = (x[-1] - x[0]) / (M - 1)
print(f'The space grid is of width {dx}')
print(f'The temporal grid is of width {timestep}')

The space grid is of width 0.2
The temporal grid is of width 0.0025


We generate training data from initial conditions

$$u(x,0)=\sum_{l=1}^2\left( \frac{12}{\eta}c_l^2 \right)sech^2\left(c_l\left(\left( x+\frac{P}{2}-d_lP \right)-\frac{P}{2}\right)\right) $$

where $c_1,c_2$ and $d_1,d_2$ are randomly drawn from the uniform distributions $\mathcal{U}(0.5,2)$ and $\mathcal{U}(0,1)$ respectively. The initial state are two waves of height $2c_1^2$ and $2c_2^2$ centred at $d_1P$ and $d_2P$, with periodicity imposed.

In [3]:
# Define the initial condition sampler.
n_solitons = 2

def initial_condition() -> Callable[[np.random.Generator], np.ndarray]:
    P = (x[-1] - x[0]) * M / (M - 1)
    sech = lambda a: 1 / np.cosh(a)

    def sampler_solitons(rng: np.random.Generator) -> np.ndarray:
        ks = rng.uniform(0.5, 2.0, n_solitons)
        ds = rng.uniform(0.0, 1.0, n_solitons)
        u0 = np.zeros_like(x)

        for k, d in zip(ks, ds):
            center = (x + P / 2 - d * P) % P - P / 2
            coeff = 12.0 / eta * k**2
            u0 += coeff * sech(np.abs(k * center)) ** 2

        #u0 = np.concatenate([u0[M:], u0[:M]], axis=-1)
        return u0
    
    return sampler_solitons

The KdV equation admits the following Hamiltonian functional

$$ \mathcal{H}[u]=-\int_{\Omega}\left(\frac{\eta}{6}u^3+\frac{\gamma^2}{2}u_x^2 \right)dx $$

During simulation and after defining our spatial discretization, this integral is approximated by 

$$ \mathcal{H}[u]\approx -h\sum_{i=0}^{M-1}\left(\frac{\eta}{6}u_i^3+\frac{\gamma^2}{2}(\delta_fu_i)² \right) $$

where the operator $\delta_f$ denotes forward difference, i.e. $\delta_fu_i=(u_{i+1}-u_{i-1})/(2h)$ 

$$S_d=\frac{1}{2h}\begin{bmatrix} 0 & 1 & 0 & \cdots & 0 & -1 \\ -1 & 0 & 1 & 0 & \cdots & \\ 0 & -1 & 0 & 1 & 0 & \cdots \\ & & \ddots & \ddots & \ddots & \\ & \cdots & 0 & -1 & 0 & 1 \\ 1 & 0 & \cdots & 0 & -1 & 0  \end{bmatrix} $$

where the first and last rows are adjusted according to the periodic boundary conditions.

In [6]:
# 1st order forward difference matrix.
Dp = diags_array([1, -1, 1], offsets=[-M + 1, 0, 1], shape=(M, M)).toarray()
Dp = 1 / dx * Dp
# 1st order central difference matrix.
D1 = diags_array([1, -1, 1, -1], offsets=[-M + 1, -1, 1, M - 1], shape=(M, M)).toarray()
D1 = 0.5 / dx * D1
# 2nd order central difference matrix.
D2 = diags_array([1, 1, -2, 1, 1], offsets=[-M + 1, -1, 0, 1, M - 1], shape=(M, M)).toarray()
D2 = 1 / dx**2 * D2

# Define the Hamiltonian and its gradient.
def hamiltonian(u: np.ndarray) -> np.ndarray:
    integrand = -1 / 6 * eta * u**3 + 0.5 * gamma **2 * ((u @ Dp) ** 2)
    return np.sum(integrand, axis=-1)

def hamiltonian_grad(u: np.ndarray) -> np.ndarray:
    return -0.5 * eta * u**2 - (gamma**2 * u @ D2)

Simulate the trajectory taking into account that we can rewrite the KdV equation as 

$$u_t=S\frac{\partial\mathcal{H}}{\partial u}[u]-R\frac{\partial V}{\partial u}[u]+f(u,x,t) $$

with $S=\frac{\partial}{\partial x}$, $R=0$ and $f(u,x,t)=0$.

In [8]:
traj_length = t_axis.shape[0]
x_ = np.zeros((num_trajectories, traj_length, num_spatial_points))
dxdt = np.zeros((num_trajectories, traj_length, num_spatial_points))
t = np.zeros((num_trajectories, traj_length))
u = np.zeros((num_trajectories, traj_length - 1, num_spatial_points))

for i in tqdm(range(num_trajectories)):
    # Define the initial conditions.
    rng = np.random.default_rng(42)
    x0 = initial_condition()(rng)
    # Define the time derivative of the state.
    S = D1
    x_dot = lambda t, x: hamiltonian_grad(x) @ (S.T)
    # Generate the trajectory using scipy.
    out_ivp = solve_ivp(
        fun=x_dot, t_span=(t_axis[0], t_axis[-1]), y0=x0, t_eval=t_axis, rtol=1e-10
    )
    x_[i], t[i] = out_ivp["y"].T, out_ivp["t"].T

100%|██████████| 1/1 [00:00<00:00, 48.11it/s]


In [ ]:
from tools import create_video

create_video(
    [x_[0]],
    ["Ground truth"],
    x_axis=x
)